# 06 — Confidence Gate

## Purpose
Scores each extracted document segment and routes it to either
`AUTO_APPROVE` or `HUMAN_REVIEW` based on a composite confidence score
and a set of explicit flag rules. Also catches documents that failed
earlier pipeline stages or were classified as unknown and routes them
directly to the review queue with a reason flag.

## What this notebook does
Computes a composite score for each segment from two signals — field
completeness (proportion of mandatory fields successfully extracted) and
extract certainty (average confidence across non-missing mandatory fields).
Signal weights are loaded from `DOC_TYPE_CONFIG` per doc type, defaulting
to 50/50 if not configured.

A segment is routed to `HUMAN_REVIEW` if any of the following flags fire,
regardless of composite score:

- `MISSING:<field_ids>` - one or more mandatory fields returned null;
  lists exactly which fields are missing (e.g. `MISSING:supplier_name, hts_codes`)
- `LOW_CONFIDENCE:<field_ids>` - one or more mandatory fields extracted
  with confidence below `FIELD_CONFIDENCE_THRESHOLD`; lists exactly which
  fields are affected (e.g. `LOW_CONFIDENCE:facility_address`)
- `LOW_COMPOSITE_SCORE:<score>` - composite score below `AUTO_APPROVE_THRESHOLD`
- `LOW_CLASSIFY_CONFIDENCE:<score>` - classify confidence below
  `CLASSIFY_CONF_THRESHOLD`, meaning the doc type or page boundaries
  may be wrong
- `UNKNOWN_DOC_TYPE` - classifier could not identify the document type;
  no extraction was attempted
- `PIPELINE_ERROR:<stage>` - document failed at an earlier stage (parse,
  classify, or extract); root cause investigation required

Only segments with no flags and a composite score at or above threshold
are routed to `AUTO_APPROVE`. Flag reasons are stored as a JSON array on
both `DOCUMENTS_SCORED` and `REVIEW_QUEUE` so the review portal can display
exactly why a document was flagged — including the specific field names that
are missing or low confidence, giving reviewers actionable context without
opening the source document.

Status is updated in `DOCUMENTS_INGESTED` to `AUTO_APPROVED` or
`HUMAN_REVIEW`. Audit events (`AUTO_APPROVED`, `QUEUED`) are written to
`AUDIT.REVIEW_AUDIT_LOG` for full pipeline traceability.

## Outputs
| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_SCORED` | One row per segment - composite score, signal breakdown, gate result, flag reasons with field names |
| `PROCESSING.REVIEW_QUEUE` | One row per flagged segment - doc type, score, flag reasons, notes |
| `INGEST.DOCUMENTS_INGESTED` | STATUS updated to `AUTO_APPROVED` or `HUMAN_REVIEW` |
| `AUDIT.REVIEW_AUDIT_LOG` | One event per segment - `AUTO_APPROVED` or `QUEUED` with score and model |

## Key design decisions
- **Any missing mandatory field forces review** - regardless of composite
  score; a document missing a required field cannot be auto-approved even
  if all other fields extracted with high confidence. The flag includes
  the exact field names so reviewers know immediately what to look for
- **Per-field confidence floor** - any mandatory field with confidence
  below `FIELD_CONFIDENCE_THRESHOLD` forces review regardless of composite
  score; the flag names the specific low-confidence fields so reviewers
  know which values to verify
- **Classify confidence as override signal not composite input** - low
  classify confidence flags the document for review without affecting the
  extraction score, since the two are independent failure modes
- **Pipeline errors caught here** - documents that failed parse, classify,
  or extract are routed to the review queue here rather than silently
  dropped, giving reviewers visibility into all documents that need
  attention in one place

In [ ]:
import json
import pandas as pd
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
PROCESSING_SCHEMA = 'PROCESSING'
INGEST_SCHEMA     = 'INGEST'
CONFIG_SCHEMA     = 'CONFIG'
AUDIT_SCHEMA      = 'AUDIT'

AUTO_APPROVE_THRESHOLD    = 0.80
CLASSIFY_CONF_THRESHOLD   = 0.80   # below threshold - LOW_CLASSIFY_CONFIDENCE flag
FIELD_CONFIDENCE_THRESHOLD  = 0.80   # individual field confidence floor

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

s = get_active_session()

# Load EXTRACTION_MODEL from PIPELINE_CONFIG
config = {
    row['CONFIG_KEY']: row['CONFIG_VALUE']
    for row in s.sql(f"""
        SELECT CONFIG_KEY, CONFIG_VALUE
        FROM {DB}.{CONFIG_SCHEMA}.PIPELINE_CONFIG
        WHERE CONFIG_KEY = 'extract_model'
          AND IS_ACTIVE  = TRUE
    """).collect()
}

EXTRACTION_MODEL = config.get('extract_model')

if not EXTRACTION_MODEL:
    raise ValueError(
        "Missing 'extract_model' in PIPELINE_CONFIG — "
        "run 00_setup_config.ipynb first"
    )

info(f"Extraction model: {EXTRACTION_MODEL}")

In [ ]:
#Load confidence weights from DOC_TYPE_CONFIG
DEFAULT_WEIGHTS = {
    'field_completeness': 0.50,
    'extract_certainty':  0.50,
}

weights_cache = {}

doc_type_rows = s.sql(f"""
    SELECT DOC_TYPE, CONFIDENCE_WEIGHTS
    FROM {DB}.{CONFIG_SCHEMA}.DOC_TYPE_CONFIG
    WHERE IS_ACTIVE = TRUE
""").collect()

for row in doc_type_rows:
    doc_type = row['DOC_TYPE']
    raw      = row['CONFIDENCE_WEIGHTS']
    if raw:
        weights = json.loads(raw) if isinstance(raw, str) else raw
    else:
        weights = DEFAULT_WEIGHTS
    weights_cache[doc_type] = weights

info(f"Weights loaded for {len(weights_cache)} doc type(s)")

In [ ]:
# Pull all documents that need scoring
#  - Successfully classified and extracted documents
#  - Documents with pipeline errors (PARSE_ERROR, CLASSIFY_ERROR etc.)
#  - Documents classified as 'unknown'

# Successfully classified + extracted
extracted_docs = s.sql(f"""
    SELECT
        c.CHILD_DOC_ID,
        c.DOC_ID,
        c.DOC_TYPE,
        c.CONFIDENCE                                        AS CLASSIFY_CONFIDENCE,
        i.STATUS                                            AS INGEST_STATUS,
        SUM(CASE WHEN f.IS_MANDATORY = TRUE
                  AND f.IS_MISSING   = FALSE THEN 1 ELSE 0 END)
            / NULLIF(SUM(CASE WHEN f.IS_MANDATORY = TRUE THEN 1 END), 0)
                                                            AS FIELD_COMPLETENESS_RAW,
        AVG(CASE WHEN f.IS_MANDATORY = TRUE
                  AND f.IS_MISSING   = FALSE
             THEN f.FIELD_CONFIDENCE END)                   AS EXTRACT_CERTAINTY_RAW,
        SUM(CASE WHEN f.IS_MANDATORY = TRUE
                  AND f.IS_MISSING   = TRUE THEN 1 ELSE 0 END)
                                                            AS MISSING_MANDATORY_COUNT,
        COUNT(CASE WHEN f.IS_MANDATORY = TRUE THEN 1 END)  AS TOTAL_MANDATORY,
        -- Missing field names 
        LISTAGG(
            CASE WHEN f.IS_MANDATORY = TRUE
                  AND f.IS_MISSING   = TRUE
             THEN f.FIELD_ID END, ', '
        ) WITHIN GROUP (ORDER BY f.FIELD_ID)                AS MISSING_FIELD_IDS,
        -- Low confidence field names
        SUM(CASE WHEN f.FIELD_CONFIDENCE < {FIELD_CONFIDENCE_THRESHOLD}
             THEN 1 ELSE 0 END)                             AS LOW_CONF_FIELD_COUNT,
        LISTAGG(
            CASE WHEN f.FIELD_CONFIDENCE < {FIELD_CONFIDENCE_THRESHOLD}
             THEN f.FIELD_ID END, ', '
        ) WITHIN GROUP (ORDER BY f.FIELD_CONFIDENCE)        AS LOW_CONF_FIELD_IDS
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
    JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON c.DOC_ID = i.DOC_ID
    JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_EXTRACTED_FLAT f
        ON  c.CHILD_DOC_ID     = f.CHILD_DOC_ID
        AND f.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_SCORED sc
        ON  c.CHILD_DOC_ID     = sc.CHILD_DOC_ID
        AND sc.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    WHERE sc.CHILD_DOC_ID IS NULL
    GROUP BY
        c.CHILD_DOC_ID, c.DOC_ID,
        c.DOC_TYPE, c.CONFIDENCE, i.STATUS
""").collect()

# Documents with pipeline errors
error_docs = s.sql(f"""
    SELECT
        DOC_ID          AS DOC_ID,
        STATUS          AS INGEST_STATUS,
        ORIGINAL_FILENAME
    FROM {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
    WHERE STATUS IN ('PARSE_ERROR', 'CLASSIFY_ERROR', 'EXTRACT_ERROR')
""").collect()

info(f"Extracted docs to score  : {len(extracted_docs)}")
info(f"Error docs to route      : {len(error_docs)}")


# Documents classified as unknown
unknown_docs = s.sql(f"""
    SELECT
        c.CHILD_DOC_ID,
        c.DOC_ID,
        c.DOC_TYPE,
        c.CONFIDENCE    AS CLASSIFY_CONFIDENCE,
        i.ORIGINAL_FILENAME
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
    JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON c.DOC_ID = i.DOC_ID
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.REVIEW_QUEUE r
        ON c.CHILD_DOC_ID = r.CHILD_DOC_ID
    WHERE c.DOC_TYPE    = 'unknown'
      AND r.CHILD_DOC_ID IS NULL   -- not already in review queue
""").collect()

info(f"Unknown doc type to route: {len(unknown_docs)}")

In [ ]:

scored_rows      = []
review_rows      = []
auto_approve_ids = []

# Score extracted documents
for row in extracted_docs:
    child_doc_id          = row['CHILD_DOC_ID']
    doc_type              = row['DOC_TYPE']
    classify_confidence   = row['CLASSIFY_CONFIDENCE'] or 0.0
    missing_mandatory     = row['MISSING_MANDATORY_COUNT'] or 0
    total_mandatory       = row['TOTAL_MANDATORY'] or 0
    ingest_status         = row['INGEST_STATUS']
    missing_field_ids   = row['MISSING_FIELD_IDS'] or ''      
    low_conf_count      = row['LOW_CONF_FIELD_COUNT'] or 0    
    low_conf_fields     = row['LOW_CONF_FIELD_IDS'] or ''     
    
    # Field completeness — ratio of mandatory fields present
    if total_mandatory > 0:
        field_completeness = (total_mandatory - missing_mandatory) / total_mandatory
    else:
        field_completeness = 1.0

    # Extract certainty — avg confidence on non-missing mandatory fields
    extract_certainty = row['EXTRACT_CERTAINTY_RAW'] or 0.0

    # Weights from config
    weights = weights_cache.get(doc_type, DEFAULT_WEIGHTS)
    w_comp  = weights.get('field_completeness', 0.50)
    w_cert  = weights.get('extract_certainty',  0.50)

    # Composite score
    composite = (field_completeness * w_comp) + (extract_certainty * w_cert)

    # Flag reasons 
    flags = []

    if doc_type == 'unknown':
        flags.append('UNKNOWN_DOC_TYPE')

    if classify_confidence < CLASSIFY_CONF_THRESHOLD:
        flags.append(f'LOW_CLASSIFY_CONFIDENCE:{round(classify_confidence, 3)}')

    if missing_mandatory > 0:
        missing_list = missing_field_ids if missing_field_ids else 'unknown'
        flags.append(f'MISSING:{missing_list}')
    
    if low_conf_count > 0:
        low_list = low_conf_fields if low_conf_fields else 'unknown'
        flags.append(f'LOW_CONFIDENCE:{low_list}')
    
    if composite < AUTO_APPROVE_THRESHOLD:
        flags.append(f'LOW_COMPOSITE_SCORE:{round(composite, 3)}')


    # Gate decision
    # Route to HUMAN_REVIEW if ANY flag exists
    # Auto-approve only when completely clean
    gate_result = 'HUMAN_REVIEW' if flags else 'AUTO_APPROVE'

    scored_rows.append({
        'CHILD_DOC_ID':        child_doc_id,
        'EXTRACTION_MODEL':    EXTRACTION_MODEL,
        'CLASSIFY_CONFIDENCE': classify_confidence,
        'FIELD_COMPLETENESS':  round(field_completeness, 4),
        'EXTRACT_CERTAINTY':   round(extract_certainty, 4),
        'COMPOSITE_SCORE':     round(composite, 4),
        'GATE_RESULT':         gate_result,
        'FLAG_REASONS':        json.dumps(flags),
    })

    if gate_result == 'HUMAN_REVIEW':
        review_rows.append({
            'CHILD_DOC_ID':  child_doc_id,
            'DOC_TYPE':      doc_type,
            'COMPOSITE_SCORE': round(composite, 4),
            'FLAG_REASONS':  json.dumps(flags),
            'NOTES':         f"Routed by confidence gate. Flags: {', '.join(flags)}",
            'STATUS':        'PENDING',
        })
    else:
        auto_approve_ids.append(child_doc_id)

    # Log per document
    flag_str = ', '.join(flags) if flags else '✓ clean'
    info(f"  [{gate_result}] {child_doc_id} ({doc_type}) "
         f"— composite: {composite:.3f} | {flag_str}")
         
# Route unknown doc type documents
for row in unknown_docs:
    child_doc_id        = row['CHILD_DOC_ID']
    classify_confidence = row['CLASSIFY_CONFIDENCE'] or 0.0
    filename            = row['ORIGINAL_FILENAME']

    flags = ['UNKNOWN_DOC_TYPE']
    if classify_confidence < CLASSIFY_CONF_THRESHOLD:
        flags.append(f'LOW_CLASSIFY_CONFIDENCE:{round(classify_confidence, 3)}')

    review_rows.append({
        'CHILD_DOC_ID':   child_doc_id,
        'DOC_TYPE':       'unknown',
        'COMPOSITE_SCORE': None,
        'FLAG_REASONS':   json.dumps(flags),
        'NOTES':          f"Document type could not be identified for "
                          f"'{filename}'. Manual classification required.",
        'STATUS':         'PENDING',
    })
    warning(f"  [UNKNOWN] {filename} — classify confidence: {classify_confidence:.3f}")

# Route pipeline error documents
for row in error_docs:
    parent_doc_id = row['DOC_ID']
    status        = row['INGEST_STATUS']
    filename      = row['ORIGINAL_FILENAME']

    flag = {
        'PARSE_ERROR':    'PIPELINE_ERROR:PARSE_FAILED',
        'CLASSIFY_ERROR': 'PIPELINE_ERROR:CLASSIFY_FAILED',
        'EXTRACT_ERROR':  'PIPELINE_ERROR:EXTRACT_FAILED',
    }.get(status, f'PIPELINE_ERROR:{status}')

    review_rows.append({
        'CHILD_DOC_ID':   parent_doc_id,
        'DOC_TYPE':       None,
        'COMPOSITE_SCORE': None,
        'FLAG_REASONS':   json.dumps([flag]),
        'NOTES':          f"Pipeline error on '{filename}' — status: {status}. "
                          f"Manual investigation required.",
        'STATUS':         'PENDING',
    })
    warning(f"  [PIPELINE ERROR] {filename} — {flag}")

In [ ]:
# Write DOCUMENTS_SCORED
if scored_rows:
    s.write_pandas(
        pd.DataFrame(scored_rows),
        table_name='DOCUMENTS_SCORED',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(scored_rows)} row(s) to DOCUMENTS_SCORED")

# Write REVIEW_QUEUE - all sources together
if review_rows:
    s.write_pandas(
        pd.DataFrame(review_rows),
        table_name='REVIEW_QUEUE',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(review_rows)} row(s) to REVIEW_QUEUE "
         f"({len([r for r in review_rows if r['DOC_TYPE'] == 'unknown'])} unknown, "
         f"{len([r for r in review_rows if r.get('FLAG_REASONS', '').find('PIPELINE_ERROR') > -1])} errors, "
         f"{len([r for r in review_rows if r['DOC_TYPE'] not in (None, 'unknown')])} low score/missing)")


In [ ]:
# Update STATUS in DOCUMENTS_INGESTED

if auto_approve_ids:
    # Get parent DOC_IDs for auto-approved child docs
    id_list = ','.join(f"'{i}'" for i in auto_approve_ids)
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'AUTO_APPROVED'
        WHERE DOC_ID IN (
            SELECT DOC_ID
            FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED
            WHERE CHILD_DOC_ID IN ({id_list})
        )
    """).collect()
    info(f"Updated {len(auto_approve_ids)} document(s) to AUTO_APPROVED")

review_child_ids = [
    r['CHILD_DOC_ID'] for r in review_rows
    if r['DOC_TYPE'] is not None   # exclude pipeline error docs
]
if review_child_ids:
    id_list = ','.join(f"'{i}'" for i in review_child_ids)
    s.sql(f"""
        UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED
        SET STATUS = 'HUMAN_REVIEW'
        WHERE DOC_ID IN (
            SELECT DOC_ID
            FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED
            WHERE CHILD_DOC_ID IN ({id_list})
        )
    """).collect()
    info(f"Updated {len(review_child_ids)} document(s) to HUMAN_REVIEW")

In [ ]:
# Write Review_Audit_Log

s.sql(f"""
    INSERT INTO {DB}.{AUDIT_SCHEMA}.REVIEW_AUDIT_LOG
        (CHILD_DOC_ID, DOC_ID, ORIGINAL_FILENAME, STAGE_PATH, DOC_TYPE,
         EVENT_TYPE, ACTIVITY_NOTES, PERFORMED_BY, EVENT_AT)
    SELECT
        sc.CHILD_DOC_ID,
        c.DOC_ID,
        i.ORIGINAL_FILENAME,
        i.STAGE_PATH,
        c.DOC_TYPE,
        'AUTO_APPROVED',
        'Score: ' || ROUND(sc.COMPOSITE_SCORE, 3)
            || ' | Model: ' || sc.EXTRACTION_MODEL
            -- include informational flags (e.g. CROSS_DOC) when present
            || CASE
                   WHEN ARRAY_SIZE(TRY_PARSE_JSON(sc.FLAG_REASONS::VARCHAR)) > 0
                   THEN ' | Flags: ' ||
                        ARRAY_TO_STRING(TRY_PARSE_JSON(sc.FLAG_REASONS::VARCHAR), '; ')
                   ELSE ''
               END,
        'pipeline',
        sc.SCORED_AT
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_SCORED sc
    JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        ON sc.CHILD_DOC_ID = c.CHILD_DOC_ID
    LEFT JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON c.DOC_ID = i.DOC_ID
    LEFT JOIN {DB}.{AUDIT_SCHEMA}.REVIEW_AUDIT_LOG al
        ON  sc.CHILD_DOC_ID = al.CHILD_DOC_ID
        AND al.EVENT_TYPE   = 'AUTO_APPROVED'
    WHERE sc.GATE_RESULT      = 'AUTO_APPROVE'
      AND sc.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
      AND al.CHILD_DOC_ID IS NULL
""").collect()

info("Inserted AUTO_APPROVED events into REVIEW_AUDIT_LOG")

s.sql(f"""
    INSERT INTO {DB}.{AUDIT_SCHEMA}.REVIEW_AUDIT_LOG
        (CHILD_DOC_ID, DOC_ID, ORIGINAL_FILENAME, STAGE_PATH, DOC_TYPE,
         EVENT_TYPE, ACTIVITY_NOTES, PERFORMED_BY, EVENT_AT)
    SELECT
        rq.CHILD_DOC_ID,
        c.DOC_ID,
        i.ORIGINAL_FILENAME,
        i.STAGE_PATH,
        rq.DOC_TYPE,
        'QUEUED',
        -- same readable format as AUTO_APPROVED (score, model, then flags)
        CASE WHEN sc.COMPOSITE_SCORE IS NOT NULL
             THEN 'Score: ' || ROUND(sc.COMPOSITE_SCORE, 3) || ' | '
             ELSE '' END
            || CASE WHEN sc.EXTRACTION_MODEL IS NOT NULL
                    THEN 'Model: ' || sc.EXTRACTION_MODEL || ' | '
                    ELSE '' END
            || 'Flags: ' ||
               COALESCE(
                   ARRAY_TO_STRING(TRY_PARSE_JSON(rq.FLAG_REASONS::VARCHAR), '; '),
                   rq.FLAG_REASONS::VARCHAR
               ),
        'pipeline',
        rq.QUEUED_AT
    FROM {DB}.{PROCESSING_SCHEMA}.REVIEW_QUEUE rq
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        ON rq.CHILD_DOC_ID = c.CHILD_DOC_ID
    LEFT JOIN {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
        ON c.DOC_ID = i.DOC_ID
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_SCORED sc
        ON  rq.CHILD_DOC_ID     = sc.CHILD_DOC_ID
        AND sc.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    LEFT JOIN {DB}.{AUDIT_SCHEMA}.REVIEW_AUDIT_LOG al
        ON  rq.CHILD_DOC_ID = al.CHILD_DOC_ID
        AND al.EVENT_TYPE   = 'QUEUED'
    WHERE al.CHILD_DOC_ID IS NULL
""").collect()

info("Inserted QUEUED events into REVIEW_AUDIT_LOG")

In [ ]:
# Summary
auto_count    = len(auto_approve_ids)
error_count   = len(error_docs)
unknown_count = len(unknown_docs)
scored_count  = len(scored_rows)
review_count  = len([r for r in review_rows
                     if r['DOC_TYPE'] not in (None, 'unknown')])
total         = scored_count + unknown_count + error_count

print(f"\n Confidence gate summary ")
print(f"  Extraction model         : {EXTRACTION_MODEL}")
print(f"  Auto approve threshold   : {AUTO_APPROVE_THRESHOLD}")
print(f"  Field conf threshold     : {FIELD_CONFIDENCE_THRESHOLD}")
print(f"  ───────────────────────────────────────────────────────────────────")
print(f"  Extracted docs scored    : {scored_count}")
print(f"    └─ Auto approved       : {auto_count}"
      + (f" ({auto_count/scored_count*100:.1f}% of scored)" if scored_count else ""))
print(f"    └─ Routed to review    : {review_count}"
      + (f" ({review_count/scored_count*100:.1f}% of scored)" if scored_count else ""))
print(f"  Unknown doc type         : {unknown_count}")
print(f"  Pipeline errors          : {error_count}")
print(f"  ───────────────────────────────────────────────────────────────────")
print(f"  Total processed          : {total}")
print(f"  Total routed to review   : {len(review_rows)}"
      + (f" ({len(review_rows)/total*100:.1f}% of total)" if total else ""))

print(f"\n Score distribution")
s.sql(f"""
    SELECT
        c.DOC_TYPE,
        sc.GATE_RESULT,
        COUNT(*)                              AS DOC_COUNT,
        ROUND(AVG(sc.COMPOSITE_SCORE), 3)    AS AVG_COMPOSITE,
        ROUND(MIN(sc.COMPOSITE_SCORE), 3)    AS MIN_COMPOSITE,
        ROUND(MAX(sc.COMPOSITE_SCORE), 3)    AS MAX_COMPOSITE,
        ROUND(AVG(sc.FIELD_COMPLETENESS), 3) AS AVG_COMPLETENESS,
        ROUND(AVG(sc.EXTRACT_CERTAINTY), 3)  AS AVG_CERTAINTY
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_SCORED sc
    JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_CLASSIFIED c
        ON sc.CHILD_DOC_ID = c.CHILD_DOC_ID
    WHERE sc.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    GROUP BY c.DOC_TYPE, sc.GATE_RESULT
    ORDER BY c.DOC_TYPE, sc.GATE_RESULT
""").show()

print(f"\n Flag reason breakdown")
s.sql(f"""
    SELECT
        SPLIT_PART(f.value::VARCHAR, ':', 1)    AS FLAG_TYPE,
        f.value::VARCHAR                         AS FLAG_DETAIL,
        COUNT(*)                                 AS OCCURRENCES
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_SCORED sc,
    LATERAL FLATTEN(input => PARSE_JSON(sc.FLAG_REASONS)) f
    WHERE sc.EXTRACTION_MODEL = '{EXTRACTION_MODEL}'
    GROUP BY f.value::VARCHAR
    ORDER BY FLAG_TYPE, OCCURRENCES DESC
""").show()

print(f"\n Review queue breakdown")
s.sql(f"""
    SELECT
        DOC_TYPE,
        STATUS,
        SPLIT_PART(f.value::VARCHAR, ':', 1)    AS FLAG_TYPE,
        COUNT(*)                                 AS QUEUE_COUNT
    FROM {DB}.{PROCESSING_SCHEMA}.REVIEW_QUEUE rq,
    LATERAL FLATTEN(input => PARSE_JSON(rq.FLAG_REASONS)) f
    GROUP BY DOC_TYPE, STATUS, FLAG_TYPE
    ORDER BY QUEUE_COUNT DESC
""").show()

print(f"\n REVIEW_AUDIT_LOG summary ")
s.sql(f"""
    SELECT
        EVENT_TYPE,
        COUNT(*)                        AS EVENT_COUNT,
        COUNT(DISTINCT CHILD_DOC_ID)    AS DOCS_AFFECTED
    FROM {DB}.{AUDIT_SCHEMA}.REVIEW_AUDIT_LOG
    GROUP BY EVENT_TYPE
    ORDER BY EVENT_COUNT DESC
""").show()